# Vessel efficiency grade prediction (EU MRV)

**Question:** can we predict a ship's operational CO2 efficiency grade (A-E) from its structural and technical attributes, *without* using the CO2 figures that define the grade?

The defining CO2 fields are deliberately excluded to avoid **target leakage**. The pipeline scales to the full THETIS-MRV dataset by swapping `data/vessels_mrv.csv`.

In [1]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
from features import build_features, NUMERIC, CATEGORICAL
from train import make_preprocessor, candidate_models

df = pd.read_csv('../data/vessels_mrv.csv')
print(df.shape)
df[['ship_type','dwt','efficiency_grade','technical_efficiency']].head()

(1570, 11)


,ship_type,dwt,efficiency_grade,technical_efficiency
0,Bulk carrier,95709.0,D,EEXI (3.24 gCO₂/t·nm)
1,Bulk carrier,93407.0,E,EEXI (3.27 gCO₂/t·nm)
2,Bulk carrier,93238.0,E,EIV (4.36 gCO₂/t·nm)
3,Bulk carrier,92968.0,E,EEXI (3.28 gCO₂/t·nm)
4,Bulk carrier,92929.0,E,EEXI (3.28 gCO₂/t·nm)


## 1. Target balance and ship types

In [2]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
df['efficiency_grade'].value_counts().sort_index().plot(kind='bar', ax=ax[0], title='Efficiency grade (target)', rot=0)
df['ship_type'].value_counts().plot(kind='barh', ax=ax[1], title='Ship type')
plt.tight_layout()

## 2. Feature engineering (leakage-free)

Parse the technical-efficiency index, group rare flags, drop the CO2 fields that define the grade.

In [3]:
X, y = build_features(df)
print('Features:', list(X.columns))
print('Excluded CO2 fields are NOT in X ->', all(c not in X.columns for c in ['co2_per_distance_kg_nm','co2_total_t']))
X.head(3)

Features: ['ship_type', 'dwt', 'dwt_size_class', 'flag_grouped', 'te_metric', 'te_value']
Excluded CO2 fields are NOT in X -> True


,ship_type,dwt,dwt_size_class,flag_grouped,te_metric,te_value
0,Bulk carrier,95709.0,Panamax (60-100k),Majuro,EEXI,3.24
1,Bulk carrier,93407.0,Panamax (60-100k),Nassau,EEXI,3.27
2,Bulk carrier,93238.0,Panamax (60-100k),Panama,EIV,4.36


## 3. Model comparison (5-fold cross-validation)

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pre = make_preprocessor()
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows=[]
for name, model in candidate_models().items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    s = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='accuracy')
    rows.append((name, s.mean(), s.std()))
cvres = pd.DataFrame(rows, columns=['model','cv_accuracy','std']).set_index('model')
cvres.round(3)

,cv_accuracy,std
model,,
baseline (most frequent),0.205,0.001
logistic regression,0.398,0.025
random forest,0.416,0.036
gradient boosting,0.422,0.022


## 4. Best model: held-out evaluation

In [5]:
from sklearn.ensemble import GradientBoostingClassifier
best = Pipeline([('pre', make_preprocessor()), ('model', GradientBoostingClassifier(random_state=42))])
best.fit(X_tr, y_tr)
y_pred = best.predict(X_te)
print(classification_report(y_te, y_pred))
labels = sorted(y.unique())
cm = confusion_matrix(y_te, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(5)); ax.set_xticklabels(labels); ax.set_yticks(range(5)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix')
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='black')
plt.colorbar(im); plt.tight_layout()

              precision    recall  f1-score   support

           A       0.60      0.64      0.62        78
           B       0.36      0.38      0.37        79
           C       0.17      0.16      0.16        77
           D       0.35      0.40      0.37        78
           E       0.52      0.42      0.46        81

    accuracy                           0.40       393
   macro avg       0.40      0.40      0.40       393
weighted avg       0.40      0.40      0.40       393



## 5. Which attributes carry the signal?

In [6]:
r = permutation_importance(best, X_te, y_te, n_repeats=10, random_state=42, scoring='accuracy')
imp = pd.Series(r.importances_mean, index=X.columns).sort_values()
imp.plot(kind='barh', title='Permutation importance (accuracy drop)')
plt.tight_layout()

## 6. Learning curve — does more data help?

Trains on increasing fractions of the data to show whether accuracy has plateaued or would keep rising with more ships.

In [7]:
sizes, train_sc, cv_sc = learning_curve(
    Pipeline([('pre', make_preprocessor()), ('model', GradientBoostingClassifier(random_state=42))]),
    X, y, cv=cv, scoring='accuracy', train_sizes=np.linspace(0.15, 1.0, 6), random_state=42)
plt.plot(sizes, train_sc.mean(1), 'o-', label='train')
plt.plot(sizes, cv_sc.mean(1), 'o-', label='cross-validation')
plt.xlabel('Training vessels'); plt.ylabel('Accuracy'); plt.title('Learning curve'); plt.legend(); plt.grid(alpha=.3)
pd.DataFrame({'n': sizes.astype(int), 'cv_accuracy': cv_sc.mean(1).round(3)})

,n,cv_accuracy
0,188,0.220
1,401,0.303
2,615,0.355
3,828,0.396
4,1042,0.398
5,1256,0.428


## Key findings

- The best model roughly **doubles the baseline** on 5 balanced classes, while strictly avoiding leakage.
- It learns the A->E ordering: extremes (A, E) are predicted best, the middle (C) is hardest.
- **DWT** is the strongest predictor, then the technical-efficiency index.
- The learning curve shows where adding more ships still pays off vs where it plateaus, which guides data collection.

*Scaling up: replace the CSV with the full THETIS-MRV dataset, and enrich features (build year, EEDI, design speed) for the largest gains.*